In [82]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import split, size, col, when
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DateType, FloatType, LongType

spark = SparkSession.builder\
        .appName("e-commerce_silver_transform")\
        .master("local[*]")\
        .config("spark.executor.memory", "2g")\
        .config("spark.driver.memory", "2g")\
        .getOrCreate()

In [83]:
data_path = "/home/lpascual/Projects/E-Commerce_DWH/data/bronze/parquet/event_time=2019-10-01/"
output_dir = "/home/lpascual/Projects/E-Commerce_DWH/data/silver/"

In [84]:
schema = StructType([
    StructField("event_time", DateType()),
    StructField("event_type", StringType()),
    StructField("product_id", IntegerType()),
    StructField("category_id", LongType()),
    StructField("category_code", StringType()),
    StructField("brand", StringType()),
    StructField("price", FloatType()),
    StructField("user_id", LongType()),
    StructField("user_session", StringType()),
    StructField("name", StringType()),
    StructField("username", StringType()),
    StructField("email", StringType()),
    StructField("address", StringType()),
    StructField("city", StringType()),
    StructField("country", StringType()),
    StructField("state_code", StringType()),
    StructField("zip_code", StringType()),
    StructField("sex", StringType()),
    StructField("product_name", StringType()),
    StructField("product_description", StringType())
])

product_schema = StructType([
    StructField("product_id", IntegerType()),
    StructField("product_name", StringType()),
    StructField("brand", StringType()),    
    StructField("product_description", StringType()),
    StructField("primary_category", StringType()),
    StructField("secondary_category", StringType()),
    StructField("tertiary_category", StringType())
])

event_schema = StructType([
    StructField("event_time", DateType()),
    StructField("event_type", StringType()),
    StructField("product_id", IntegerType()),
    StructField("price", FloatType()),    
    StructField("user_id", LongType()),
    StructField("user_session", StringType())
])

user_schema = StructType([
    StructField("user_id", LongType()),
    StructField("first_name", StringType()),
    StructField("last_name", StringType()),    
    StructField("user_name", StringType()),
    StructField("email", StringType()),
    StructField("address", StringType()),
    StructField("city", StringType()),
    StructField("country", StringType()),
    StructField("state_code", StringType()),
    StructField("zip_code", StringType()),
    StructField("sex", StringType())
])

In [85]:
df = spark.read.parquet(data_path, schema=schema)
print("Number of partitions:", df.rdd.getNumPartitions())

Number of partitions: 12


Transformation
1) Break category_code to primary, secondary, tertiary
2) Split name to first_name, last_name

Checks
1) Confirm email is in correct format
2) Price is > 0

In [86]:
df.show(truncate=False)

+----------+----------+-------------------+----------------------------+---------+-------+---------+------------------------------------+-----------------+-------------+-------------------------+--------------------------------------------------------+------------------+-------+----------+--------+---+-----------------+-----------------------------------------------------------------------------------+
|event_type|product_id|category_id        |category_code               |brand    |price  |user_id  |user_session                        |name             |username     |email                    |address                                                 |city              |country|state_code|zip_code|sex|product_name     |product_description                                                                |
+----------+----------+-------------------+----------------------------+---------+-------+---------+------------------------------------+-----------------+-------------+-------------------

In [87]:
df_split = df.withColumn("primary_category", when(size(split(df["category_code"], "\\.")) >= 1, split(df["category_code"], "\\.").getItem(0))) \
.withColumn("secondary_category", when(size(split(df["category_code"], "\\.")) >= 2, split(df["category_code"], "\\.").getItem(1))) \
.withColumn("tertiary_category",  when(size(split(df["category_code"], "\\.")) >= 3, split(df["category_code"], "\\.").getItem(2))) \
.withColumn("first_name", split(df["name"], " ").getItem(0)) \
.withColumn("last_name", when(size(split(df["name"], " ")) > 2, split(df["name"], " ").getItem(2)).otherwise(split(df["name"], " ").getItem(1))) \

df_split.show(truncate=False)

+----------+----------+-------------------+----------------------------+---------+-------+---------+------------------------------------+-----------------+-------------+-------------------------+--------------------------------------------------------+------------------+-------+----------+--------+---+-----------------+-----------------------------------------------------------------------------------+----------------+------------------+-----------------+----------+---------+
|event_type|product_id|category_id        |category_code               |brand    |price  |user_id  |user_session                        |name             |username     |email                    |address                                                 |city              |country|state_code|zip_code|sex|product_name     |product_description                                                                |primary_category|secondary_category|tertiary_category|first_name|last_name|
+----------+----------+---------------

In [88]:
spark.stop()